# Phase A — thang so sanh nguon prompt

Do xem LLM co thay duoc chuyen gia trong viec viet prompt khong, va **do luon toc
do**: so luot goi Grounding DINO = `1 object + so prompt`, ma DINO chiem 72%
thoi gian (xem `docs/report-phase-b.md` muc 6b).

| Muc | `--prompt-source` | Prompt | Vai tro |
|---|---|---|---|
| P0 | `generic` | `"defect."` | San tuyet doi |
| P1 | `general` | 3 general_prompts | San co san |
| P2b | `llm` | LLM, chi biet ten class | **Dong gop** |
| P2v | `llm` | LLM + anh normal | **Dong gop** |
| P3 | `manual` | 3 general + K manual | Tran (oracle) |

**P3 da chay roi** — chinh la `run_lite2` o Buoc 2. Khong ton them GPU.

**P2b va P2v chua chay duoc**: can `tools/gen_prompts.py` (Task 4 cua plan).
Notebook nay hien chay P0 va P1, von **khong can LLM**.

## Vi sao chay P0 va P1 truoc

Ba cau hoi, tra loi het trong ~8 phut mot class:

1. **Thang co gian khong?** P1 sat P3 thi tien de Phase A yeu — LLM khong con
   may cho de chung minh.
2. **`t_dino` co giam theo so prompt that khong?** P0 dung 1 prompt so voi 6 cua
   P3. Khong giam gan 6 lan thi gia dinh "so luot DINO ty le thuan voi so prompt"
   sai, va toan bo lap luan toc do phai tinh lai.
3. **May moc chay dung chua** — truoc khi dung toi VLM.

Cau hinh khoa o **lite2** (MobileSAM + MobileNetV3 + Grounding DINO), thang Buoc 1.

Notebook khac: `Benchmark_SAA.ipynb` (baseline), `Profiling_SAA_Lite.ipynb`
(Buoc 1), `Benchmark_SAA_Lite2.ipynb` (Buoc 2).

In [ ]:
# Cai dat. An toan khi chay lai nhieu lan.
%cd /content

# Xoa clone cu TRUOC khi clone. Khong co dong nay thi git clone bao
# "destination path already exists", bo qua im lang, va ban chay tiep bang
# code cu ma khong biet.
!rm -rf /content/Segment-Any-Anomaly
!git clone -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/

# setuptools >= 80 da bo lenh `setup.py develop`, ma pip dung dung lenh do cho
# ban editable khi co --no-build-isolation. Colab nang image len la GroundingDINO
# gay voi "python setup.py develop did not run successfully". Ghim lai truoc.
!pip install -q "setuptools<80" wheel
import setuptools
print('setuptools:', setuptools.__version__)

# Go pin transformers<4.36 cua GroundingDINO. Phai quet CA requirements.txt,
# khong chi *.py: pip doc requirements.txt, va bo sot no thi pip ha transformers
# xuong 4.35 (keo theo huggingface_hub va tokenizers), roi lenh pip cuoi cell
# lai day len 5.x - vong xoay do de lai mot dong conflict gia.
# Code GroundingDINO trong repo nay da duoc va cho transformers 5.x tu truoc.
import re, pathlib

for pattern in ('*.py', '*.txt'):
    for p in pathlib.Path('GroundingDINO').rglob(pattern):
        txt = p.read_text()
        patched = re.sub(r'transformers[^"\'\n]*<4\.\d+(\.\d+)?', 'transformers>=4.41.0', txt)
        if patched != txt:
            p.write_text(patched)
            print('go pin transformers trong', p)

# KHONG dat -q cho hai lenh editable duoi day: day la cho de gay nhat, va
# loi that nam trong phan output ma -q nuot mat.
%cd GroundingDINO/
!pip install -e . --no-build-isolation
%cd ../SAM
!pip install -e .
%cd ..

!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru

# KHONG kiem tra import o day. Ban editable ghi duong dan vao mot file .pth,
# ma .pth chi duoc doc luc interpreter khoi dong - package vua cai xong van
# "khong ton tai" voi kernel dang chay. Kiem tra nam o cell sau lenh restart.
print('\nCai dat xong. Chay cell tiep theo de restart runtime,')
print('roi cell sau do se xac nhan package da cai duoc that.')

In [ ]:
# Restart so updated transformers is loaded from disk
import os
os.kill(os.getpid(), 9)

In [ ]:
# Xac nhan package da cai THAT - chay sau restart, vi ban editable chi hien
# ra voi interpreter khoi dong lai.
%cd /content/Segment-Any-Anomaly
import importlib.util

missing = [m for m in ('groundingdino', 'segment_anything')
           if importlib.util.find_spec(m) is None]

for m in ('groundingdino', 'segment_anything'):
    print(f'{m}: {"THIEU" if m in missing else "OK"}')

if missing:
    raise RuntimeError(
        f'Cai dat that bai: {missing}. DUNG chay tiep - moi class se chet o dong '
        f'import. Doc output pip cua cell 1 de biet la loi setuptools hay loi '
        f'bien dich CUDA extension.'
    )

%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd weights
# -nc: co file roi thi bo qua. Khong co no thi chay lai cell se tai lai 2.4 GB
# va de ra sam_vit_h_4b8939.pth.1 - mot ban sao vo dung.
!wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -nc -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

from google.colab import userdata
import json, pathlib, os

pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
pathlib.Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY')
}))
!chmod 600 /root/.kaggle/kaggle.json

# Accept dataset terms first at: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/datasets/ --unzip

os.environ['MVTEC_DIR'] = '/content/datasets'

from datasets import mvtec_classes
present = [c for c in mvtec_classes if os.path.isdir(f'/content/datasets/{c}')]
print(f'MVTec: {len(present)}/15 classes ready:', present)

## Cai MobileSAM

Bat buoc cho cau hinh lite2.

In [ ]:
# MobileSAM cho Lite-1 va Lite-2. No giu nguyen interface SamPredictor nen la
# drop-in that su - SAA/backbones.py khong phai doi gi ngoai ten bien the.
%cd /content/Segment-Any-Anomaly
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git
!wget -q -P weights/ https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt

import os, importlib.util

ckpt = 'weights/mobile_sam.pt'
size_mb = os.path.getsize(ckpt) / 1e6 if os.path.exists(ckpt) else 0
print(f'{ckpt}: {"OK, %.0f MB" % size_mb if size_mb > 1 else "THIEU - kiem tra URL"}')
print('mobile_sam:', 'OK' if importlib.util.find_spec('mobile_sam') else 'THIEU')

# Cai them co the keo torch/transformers khac ve. Cho no gay O DAY chu dung de
# gay giua lan profiling.
try:
    from GroundingDINO.groundingdino.models import build_model
    print('GroundingDINO van OK sau khi cai')
except Exception as e:
    print(f'CANH BAO: GroundingDINO gay sau khi cai MobileSAM - {type(e).__name__}: {e}')

## Chon muc va class

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
except ValueError:
    drive.mount('/content/drive', force_remount=True)

# ---------------------------------------------------------------------------
# Bat dau bang MOT class de kiem may moc va do gian cua thang. Ra so hop ly roi
# moi mo rong ca 15 class.
# ---------------------------------------------------------------------------
CLASSES = ['carpet']
LEVELS = ['P0', 'P1']          # them 'P2b', 'P2v' khi Task 4 xong

# (prompt_source, mau duong dan file JSON hoac None)
LEVEL_CONFIG = {
    'P0':  ('generic', None),
    'P1':  ('general', None),
    'P2b': ('llm', 'SAA/prompts/generated/{dataset}-blind.json'),
    'P2v': ('llm', 'SAA/prompts/generated/{dataset}-vision.json'),
    'P3c': ('llm', 'SAA/prompts/generated/{dataset}-manual-as-json.json'),
}

DRIVE_ROOT = '/content/drive/MyDrive/SAA_results'
DATASET = 'mvtec'
os.environ['MVTEC_DIR'] = '/content/datasets'

print('CLASSES =', CLASSES)
print('LEVELS  =', LEVELS)

## Chay thang

In [ ]:
# Goi thang eval_SAA.py chu khong qua run_MVTec.py: dang chay mot tap con class,
# va moi muc can thu muc rieng.
%cd /content/Segment-Any-Anomaly
import os, subprocess, pandas as pd
from collections import deque

def already_done(root, class_name):
    path = f'{root}/csv/{DATASET}-indx-0.csv'
    if not os.path.exists(path):
        return False
    df = pd.read_csv(path, index_col=0)
    return class_name in df.index and df.loc[class_name, 'p_ap'] > 0

for level in LEVELS:
    source, file_template = LEVEL_CONFIG[level]
    root = f'{DRIVE_ROOT}/phase_a_{level}'

    for class_name in CLASSES:
        if already_done(root, class_name):
            print(f'skip {level}/{class_name}: da co ket qua')
            continue

        cmd = [
            'python', 'eval_SAA.py',
            '--dataset', DATASET, '--class-name', class_name,
            '--prompt-source', source,
            '--cal-pro', 'False',
            '--vis', 'False',
            '--sam-variant', 'mobile_sam',
            '--saliency-backbone', 'mobilenetv3',
            '--sam_checkpoint', 'weights/mobile_sam.pt',
            '--detector', 'grounding_dino',
            '--root-dir', root,
        ]
        if file_template:
            path = file_template.format(dataset=DATASET)
            if not os.path.exists(path):
                print(f'skip {level}/{class_name}: chua co {path} (can Task 4)')
                continue
            cmd += ['--llm-prompt-file', path]

        print(f'\n=== {level} / {class_name} ({source}) ===')

        # Doc output trong Python roi print: ipykernel chi bat sys.stdout o muc
        # Python, nen traceback cua tien trinh con co the bien mat hoan toan.
        proc = subprocess.Popen(cmd, cwd='/content/Segment-Any-Anomaly',
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        tail = deque(maxlen=40)
        for line in proc.stdout:
            print(line, end='')
            tail.append(line)

        if proc.wait() != 0:
            print(f'\n{"=" * 60}\nLOI: {level}/{class_name}. 40 dong cuoi:\n{"=" * 60}')
            print(''.join(tail))
            raise SystemExit(1)

## Ket qua

In [ ]:
# Doi chieu thang. P3 lay tu lan chay Buoc 2, khong chay lai.
import pandas as pd, os

COLS = ['p_ap', 'p_f1', 'r_f1_fixed', 't_dino', 't_total']

def read_level(root, label):
    path = f'{root}/csv/{DATASET}-indx-0.csv'
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, index_col=0)
    df = df[df.index.isin(CLASSES) & (df['p_ap'] > 0)]
    return df.mean(numeric_only=True).rename(label) if len(df) else None

rows = [r for r in (read_level(f'{DRIVE_ROOT}/phase_a_{lv}', lv) for lv in LEVELS)
        if r is not None]
p3 = read_level(f'{DRIVE_ROOT}/run_lite2', 'P3')
if p3 is not None:
    rows.append(p3)

if not rows:
    print('Chua co ket qua nao.')
else:
    tab = pd.DataFrame(rows)
    print(f'Trung binh tren {len(CLASSES)} class: {CLASSES}\n')
    print(tab[[c for c in COLS if c in tab]].to_string(float_format='{:.2f}'.format))

    if 'P3' in tab.index:
        print('\n--- muc tieu 1: thang co gian khong ---')
        if 'P1' in tab.index:
            gap = tab.loc['P3', 'p_f1'] - tab.loc['P1', 'p_f1']
            print(f'khoang cach P1 -> P3: {gap:+.2f} diem p_f1')
            if abs(gap) < 2:
                print('  CANH BAO: gan bang nhau. LLM khong con may cho de chung minh —')
                print('  prompt thu cong cua tac gia khong hon general_prompts bao nhieu.')
            for lv in ('P2b', 'P2v'):
                if lv in tab.index:
                    closed = (tab.loc[lv, 'p_f1'] - tab.loc['P1', 'p_f1']) / gap * 100
                    mark = 'DAT' if closed >= 50 else 'chua dat'
                    print(f'  {lv}: thu hep {closed:.1f}%  {mark}  (spec muc 7 can >= 50%)')

        print('\n--- muc tieu 2: so prompt co keo duoc t_dino xuong khong ---')
        print(f'{"muc":5s} {"t_dino":>9s} {"t_total":>9s} {"nhanh hon P3":>13s}')
        for lv in tab.index:
            print(f'{lv:5s} {tab.loc[lv, "t_dino"]:9.1f} {tab.loc[lv, "t_total"]:9.1f} '
                  f'{tab.loc["P3", "t_total"] / tab.loc[lv, "t_total"]:12.2f}x')
        print('\nP3 dung 6 prompt, P1 dung 3, P0 dung 1 (cong 1 luot object moi muc).')
        print('t_dino cua P0 khong xap xi 1/3 cua P3 thi gia dinh "so luot DINO ty le')
        print('thuan voi so prompt" SAI, va lap luan toc do cua Phase A phai tinh lai.')